In [3]:
from dotenv import load_dotenv
from datetime import datetime, timedelta

import os
import json
import openai
import requests

# Load environment variables from .env file
load_dotenv('~/Documents/Jupyter/MDST/WN25/Eventure/EventureApp/.env')

openai.api_key = os.getenv("OPENAI_API_KEY")
ticketmaster_key = os.getenv("TICKETMASTER_API_KEY")

# Verify
print("OpenAI Key Found:", bool(os.getenv("OPENAI_API_KEY")))

MODEL_NAME = "gpt-4o-mini"
client = openai.OpenAI()


OpenAI Key Found: True


Create Functions:

In [15]:
# def makeTicketmasterEventRequest(eventparams: str):
#     # Format for looking at events: https://app.ticketmaster.com/{package}/{version}/events.json?{eventparams}apikey={API key}
#     # eventparams needs to be separated by & symbols for each query. Check ticketmaster discovery page to see all query options

#     url = f'https://app.ticketmaster.com/discovery/v2/events.json?{eventparams}&apikey={ticketmaster_key}'

#     r = requests.get(url)

#     return r 

def makeTicketmasterEventRequest(params: dict):
    print(f"--- Calling Ticketmaster Event Request API with params: {params} ---")
    params['apikey'] = ticketmaster_key
    url = f'https://app.ticketmaster.com/discovery/v2/events.json'

    r = requests.get(url,params=params)
    print(f"--- Ticketmaster API Status Code: {r.status_code} ---")

    return r.text

def makeTicketmasterEventDetailsRequest(params: dict):
    params['apikey'] = ticketmaster_key
    print(f"--- Calling Ticketmaster Event Details API with params: {params} ---")
    
    url = f'https://app.ticketmaster.com/discovery/v2/events/{params['id']}.json'

    r = requests.get(url,params=params)
    print(f"--- Ticketmaster Details API Status Code: {r.status_code} ---")

    return r.text

def makeTicketmasterVenuesRequest(params: dict):
    print(f"--- Calling Ticketmaster Venues API with params: {params} ---")
    params['apikey'] = ticketmaster_key
    url = f'https://app.ticketmaster.com/discovery/v2/venues.json'

    r = requests.get(url,params=params)
    print(f"--- Ticketmaster API Status Code: {r.status_code} ---")

    return r.text

def searchTicketmasterAttractions(params: dict):
    print(f"--- Calling Ticketmaster Attractions API with params: {params} ---")
    params['apikey'] = ticketmaster_key
    url = f'https://app.ticketmaster.com/discovery/v2/attractions.json'
    
    r = requests.get(url, params=params)
    print(f"--- Ticketmaster Attractions API Status Code: {r.status_code} ---")
    
    return r.text

def makeSearchSuggestion(params: dict):
    print(f"--- Calling Ticketmaster Search Suggestions API with params: {params} ---")
    params['apikey'] = ticketmaster_key
    url = f'https://app.ticketmaster.com/discovery/v2/suggest'
    
    r = requests.get(url, params=params)
    print(f"--- Ticketmaster Search Suggestions API Status Code: {r.status_code} ---")
    return 

Describe functions to OpenAI:

In [16]:
function_descriptions = [
    {
        "type": "function",
        "function": {
            "name": "makeTicketmasterEventRequest",
            "description": "Fetches events happening in a specific location from Ticketmaster.",
            "parameters": {
                "type": "object",
                "properties": {
                    "keyword": {
                        "type": "string",
                        "description": "Search term for events, e.g., 'music', 'sports'."
                    },
                    "city": {
                        "type": "string",
                        "description": "The city to fetch events for, e.g., 'Detroit'."
                    },
                    "stateCode": {
                        "type": "string",
                        "description": "The state code, e.g., 'MI' for Michigan."
                    },
                    "countryCode": {
                        "type": "string",
                        "description": "The country code, e.g., 'US'."
                    },
                    "startDateTime": {
                        "type": "string",
                        "format": "date-time",
                        "description": "Start date and time for events in ISO 8601 format (YYYY-MM-DDTHH:MM:SSZ)."
                    },
                    "endDateTime": {
                        "type": "string",
                        "format": "date-time",
                        "description": "End date and time for events in ISO 8601 format."
                    }
                },
                "required": ["city", "stateCode", "countryCode"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "makeTicketmasterEventDetailsRequest",
            "description": "Fetching even details from an event id.",
            "parameters": {
                "type": "object",
                "properties":{
                    "id": {
                        "type": "string",
                        "description": "The id of an event, e.g., 'vvG1OZbC1JEeR-'"
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "makeTicketmasterVenuesRequest",
            "description": "Get details for a specific venue using the unique identifier for the venue.",
            "parameters": {
                "type": "object",
                "properties": {
                    "keyword": {
                        "type": "string",
                        "description": "Search term for events, e.g., 'music', 'sports'."
                    },
                    "stateCode": {
                        "type": "string",
                        "description": "The state code, e.g., 'MI' for Michigan."
                    },
                    "countryCode": {
                        "type": "string",
                        "description": "The country code, e.g., 'US'."
                    },
                },
                "required": ["stateCode", "countryCode"]
            }
        }
    },
    {
    "type": "function",
    "function": {
        "name": "searchTicketmasterAttractions",
        "description": "Searches for attractions (performers, sports teams, etc.) using the Ticketmaster API.",
        "parameters": {
            "type": "object",
            "properties": {
                "keyword": {
                    "type": "string",
                    "description": "Search term for attractions, e.g., 'Taylor Swift', 'Marvel Universe LIVE'."
                    },
                    "classificationName": {
                        "type": "string",
                        "description": "Filter attractions by classification name, e.g., 'music', 'sports'."
                    },
                    "classificationId": {
                        "type": "string",
                        "description": "Filter attractions by specific classification ID."
                    },
                    "locale": {
                        "type": "string",
                        "description": "Locale for results in 'languagecode-countrycode' format, e.g., 'en-us'."
                    },
                    "sort": {
                        "type": "string",
                        "description": "Sorting criteria for results, e.g., 'relevance,desc'."
                    }
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "makeTicketmasterEventRequest",
            "description": "Fetches events happening in a specific location from Ticketmaster.",
            "parameters": {
                "type": "object",
                "properties" : {
                    "keyword": {
                        "type": "string",
                        "description": "Keyword to search on"
                    },
                    "latlong":{
                        "type": "string",
                        "description": "Filter events by latitude and longitude, this filter is deprecated and maybe removed in a future release, please use geoPoint instead"
                    },
                    "radius":{
                        "type": "string",
                        "description": "Unit of the radius"
                    },
                    "locale":{
                        "type": "string",
                        "description": "The locale in ISO code format. Multiple comma-separated values can be provided. When omitting the country part of the code (e.g. only 'en' or 'fr') then the first matching locale is used. When using a '*' it matches all locales. '*' can only be used at the end (e.g. 'en-us,en,*')"
                    },
                    "includeTBA":{
                        "type": "string",
                        "description": "True, to include events with date to be announce (TBA). Default value is no if date parameter sent, yes otherwise. Allowed values: 'yes', 'no', or 'only'.",
                        "enum": ["yes", "no", "only"]
                    },
                    "includeTBD":{
                        "type": "string",
                        "description": "True, to include event with a date to be defined (TBD). Default value is no if date parameter sent, yes otherwise. Allowed values: 'yes', 'no', or 'only'.",
                        "enum": ["yes", "no", "only"]
                    },
                    "countryCode":{
                        "type": "string",
                        "description": "Filter suggestions by country code",
                    },
                    "countryCode":{
                        "type": "string",
                        "description": "Filter suggestions by country code",
                    },
                    "startEndDateTime": {
                        "type": "array",
                        "description": "Filter event where event start and end date overlap this range.",
                        "items": {
                            "type": "string",
                            "format": "date-time"
                        }
                    },
                    "includeSpellcheck": {
                        "type": "string",
                        "description": "yes, to include spell check suggestions in the response",
                        "enum": ["yes", "no", "only"]
                    }
                },
                "required": []
            }
        }
    }
]


In [17]:
def test_call_model(user_message: str, function_call="auto"):
    """
    1) We send user_message + function_descriptions to the model
    2) We see if it returns a function call
    3) If so, we parse arguments, call the function ourselves
    4) Return final output
    """

    completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": user_message}],
        tools=function_descriptions,  # 'functions' is now called 'tools'
        tool_choice=function_call,    # 'function_call' is now 'tool_choice'
    )
    response = completion.choices[0].message
    return response

In [18]:
def run_conversation(user_prompt_1):
  messages = []
  messages.append({"role": "user", "content": user_prompt_1})

  response_message = test_call_model(user_prompt_1)
  messages.append(response_message.model_dump(exclude_unset=True))

  tool_calls = response_message.tool_calls
  
  available_functions = {
    "makeTicketmasterEventRequest": makeTicketmasterEventRequest,
    "makeTicketmasterEventDetailsRequest": makeTicketmasterEventDetailsRequest,
    "makeTicketmasterVenuesRequest": makeTicketmasterVenuesRequest,
    "searchTicketmasterAttractions": searchTicketmasterAttractions,
    "makeSearchSuggestion": makeSearchSuggestion,
    # we can add more functions here <--
  }

  tool_responses = []

  for tool_call in tool_calls:
    function_name = tool_call.function.name
    tool_call_id = tool_call.id

    function_to_call = available_functions[function_name]

    function_args = json.loads(tool_call.function.arguments)
    function_response_data = function_to_call(params=function_args)

    tool_responses.append(
        {
          "tool_call_id": tool_call_id,
          "role": "tool",
          "name": function_name,
          "content": function_response_data,
      }
    )

  messages.extend(tool_responses)

  second_response = client.chat.completions.create(
      model=MODEL_NAME,
      messages=messages,
  )
  final_response_message = second_response.choices[0].message
  messages.append(final_response_message.model_dump(exclude_unset=True))
  final_response_content = final_response_message.content

  return final_response_content, messages

In [19]:
from datetime import date, timedelta
today = date.today()
start_date = today.strftime("%Y-%m-%dT00:00:00Z")
end_date = (today + timedelta(days=7)).strftime("%Y-%m-%dT23:59:59Z")

answer, current_history = run_conversation(
    # f"What events are going on in Michigan between {start_date} and {end_date}?",
    # f"Can you get me more details for the event with this id: vvG1OZbC1JEeR-",
    # f"What venues are available in Michigan?",
    # f"What attractions are there for jazz in Michigan from {start_date} to {end_date}",
    f"Give me some search suggestions for things going on in Ann Arbor Michigan, United States"
)
print(f"Answer: {answer}")

--- Calling Ticketmaster Event Request API with params: {'city': 'Ann Arbor', 'stateCode': 'MI', 'countryCode': 'US'} ---
--- Ticketmaster API Status Code: 200 ---
Answer: Here are some exciting events happening in Ann Arbor, Michigan:

1. **Michigan Wolverines Football vs. Washington Huskies Football**
   - **Date:** October 18, 2025
   - **Venue:** Michigan Stadium
   - [More Info & Tickets](https://www.ticketmaster.com/event/Z7r9jZ1A7oVA_)
   - ![Event Image](https://s1.ticketm.net/dam/a/43b/6d0c499b-e713-4fbc-901f-16934c19343b_1256581_CUSTOM.jpg)

2. **Sarah Millican: Late Bloomer**
   - **Date:** June 6, 2025
   - **Venue:** Michigan Theater
   - [More Info & Tickets](https://www.ticketmaster.com/sarah-millican-late-bloomer-ann-arbor-michigan-06-06-2025/event/08006139C34B3867)
   - ![Event Image](https://s1.ticketm.net/dam/a/56f/fb72cc01-333d-4edf-a6a1-96d9ca29e56f_TABLET_LANDSCAPE_LARGE_16_9.jpg)

3. **Blind Pig Comedy**
   - **Date:** April 15, 2025
   - **Venue:** Blind Pig
   

In [30]:
# Let's do a quick test with something that calls 'get_weather_info'
# user_prompt_1 = "What events are going on in Michigan for this month"
# resp = test_call_model(user_prompt_1, function_call="auto")
# print("Model response:\n", resp)


## **TODO:** Fix the OpenAI Instance 
- Fix the 'function_call=None' to be 'function_call=makeTicketmasterEventRequest'
- Assumed because 'function_descriptions' is wrong, or 'makeTicketmasterEventRequest' is broken